# 09 · CADD — a real, live deleteriousness score

Unlike the demo predictors, **CADD** is **REAL** here — fetched live from the CADD v1.7
REST API. CADD (Rentzsch et al. 2021, *Genome Medicine*, PMID 33618777) rolls dozens of
annotations into one PHRED-scaled score.

**How CADD is trained (matters for circularity):** CADD is **not** trained on clinical
ClinVar/HGMD labels. It is *proxy-supervised* — it learns to separate "observed" human-
derived variants (fixed since the human–chimp ancestor ≈ proxy-benign) from "simulated"
de-novo variants (proxy-deleterious). So its **direct** ClinVar circularity is low (unlike
REVEL). *But* CADD is a **meta-model**: CADD-Splice v1.7 folds in SpliceAI + MMSplice as
features, so leakage can enter **indirectly**. Anchor reproducibility on the **version
(v1.7, GRCh38)**, not a training date (tools/10).

This notebook also teaches the single most important practical lesson: **validate genomic
coordinates against the reference before trusting any tool.**

> ✅ **REAL / LIVE.** `fetch_cadd(...)` (defined below) calls the live CADD API. Requires network access. CADD PHRED **>= 15 ~ top 3%**, **>= 20 ~ top 1%**.

In [1]:
import sys, pathlib
# `toolkit` is THIS repo's toolkit.py (one directory up) — NOT a pip
# package and nothing to do with gnomAD. The line below puts the repo
# root on sys.path so `import toolkit` resolves to ../toolkit.py.
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import toolkit as tk
import pandas as pd, numpy as np
# %matplotlib inline is a Jupyter magic: it draws matplotlib plots inline below the cell
%matplotlib inline

## CADD — a *real* deleteriousness score, fetched live

Everything above was demo. **CADD is real.**

**CADD** (Combined Annotation Dependent Depletion; the splice-aware v1.7 is
Rentzsch *et al.* 2021, *Genome Medicine*, PMID **33618777**) rolls dozens of
annotations — conservation, regulatory marks, and splice features — into **one**
integrated deleteriousness score. The convenient number is the **PHRED-scaled**
value:

- **PHRED ≥ 15** → roughly the **top 3%** most deleterious of all possible variants
- **PHRED ≥ 20** → roughly the **top 1%**

`fetch_cadd(chrom, pos, ref, alt)` (defined in the cell below) queries the **live
CADD v1.7 REST API** and returns `{'cadd_raw': ..., 'cadd_phred': ...}` (or `None`
values on a miss). Unlike every other tool in this toolkit, CADD needs no local
file at all — there is nothing to fetch or build ahead of time, just a live
per-variant call.

> **Strand: no gotcha for CFTR.** CFTR sits on the genomic **plus (forward) strand**
> (7q31.2), so a change written on the *coding* strand (say `C>T`) appears on the
> genome as the **same** alleles (`C>T`) — you do **not** complement it. CADD is
> indexed on the plus strand, so coding-strand alleles match directly. (`fetch_cadd`
> does also try the complement as a defensive fallback, in case an upstream table
> hands you mis-oriented alleles — but for CFTR that path should never fire.)
>
> The gotcha that *does* bite is the **genome build**: GRCh37 and GRCh38 CFTR
> coordinates differ by ~ 200 kb, so a build mix-up silently matches nothing.

Let's make a **live** call on a position whose ref/alt genuinely matches the
reference genome.

> **Reproducibility.** Because CADD is queried *live*, **pin the version (v1.7, GRCh38)** and **cache the responses** so a re-run is stable and offline — a CADD version bump would change scores. Record the endpoint + version in `data_manifest.json`.

In [2]:
import time, requests


def fetch_cadd(chrom: str, pos: int, ref: str, alt: str, delay_sec: float = 0.3) -> dict:
    """Score ONE variant with the CADD v1.7 REST API (REAL, live, no local file needed).

    NOTE ON STRAND: CFTR is on the genomic PLUS strand (7q31.2), so a coding
    change (e.g. C>T) is reported on the genome as the SAME alleles (C>T) — no
    complementing is needed. This helper still falls back to trying the
    complement, purely as a guard against upstream tables that (wrongly, for
    CFTR) report minus-strand alleles; for correct input it never fires.

    API: https://cadd.gs.washington.edu/api/v1.0/GRCh38-v1.7/{chr}:{pos}-{pos}
    Returns dict(cadd_raw, cadd_phred) or None values on a miss.
    """
    url = f"https://cadd.gs.washington.edu/api/v1.0/GRCh38-v1.7/{chrom}:{pos}-{pos}"
    comp = {"A": "T", "T": "A", "C": "G", "G": "C"}
    try:
        data = requests.get(url, timeout=15).json()
    except Exception as exc:
        return {"cadd_raw": None, "cadd_phred": None, "error": str(exc)}
    for rec in data[1:] if data else []:
        if len(rec) < 6:
            continue
        r, a = rec[2], rec[3]
        if (r == ref and a == alt) or (r == comp.get(ref) and a == comp.get(alt)):
            time.sleep(delay_sec)
            return {"cadd_raw": float(rec[4]), "cadd_phred": float(rec[5])}
    time.sleep(delay_sec)
    return {"cadd_raw": None, "cadd_phred": None}

In [3]:
# LIVE call #1 — an intronic position that scores LOW (near the benign end).
res1 = fetch_cadd('7', 117548628, 'G', 'A')
print('7:117548628 G>A  ->', res1)
print(f"  CADD PHRED = {res1['cadd_phred']}  (low: near the benign end)")

7:117548628 G>A  -> {'cadd_raw': -0.964845, 'cadd_phred': 0.028}
  CADD PHRED = 0.028  (low: near the benign end)


That call **succeeded** and returned a real (low) PHRED. Now a variant that scores **high** — `c.2657+5G>A`, passed with the plus-strand alleles `G`/`A` straight from CFTR2. Because CFTR is on the **plus strand**, those are identical to the coding-strand alleles in the `c.` name, so they match the reference directly — no complementing involved.

In [4]:
# LIVE call #2 — a real high-impact splice variant at its authoritative CFTR2 coord.
res2 = fetch_cadd('7', 117602868, 'G', 'A')   # c.2657+5G>A (2789+5G>A), a donor variant
print('7:117602868 G>A  ->', res2)
phred2 = res2['cadd_phred']
if phred2 is not None:
    band = 'top ~1% (>=20)' if phred2 >= 20 else ('top ~3% (>=15)' if phred2 >= 15 else 'below 15')
    print(f'  CADD PHRED = {phred2}  ->  {band}')

7:117602868 G>A  -> {'cadd_raw': 3.907754, 'cadd_phred': 23.8}
  CADD PHRED = 23.8  ->  top ~1% (>=20)


## Example: the shared splice panel, scored by **CADD**

The same fixed panel of famous CFTR **splice** variants runs through every splice tool
(tools/07–09), so you can follow one set of variants across the series. The variant
list is `tk.A2_KNOWN_CDNA` (shared in `toolkit.py`); the scoring is shown inline below,
using CFTR2's authoritative GRCh38 coordinates.

> There is no single assembled cross-tool table function -- each notebook above scores the panel itself, inline, so you can see exactly how that tool's join works.

In [5]:
# CADD is scored LIVE, one API call per variant, over the splice panel with authoritative
# CFTR2 coords. (Deep-intronic positions can return no score -> NaN.)
cf = tk.load_cftr2()
a2 = cf[cf['cdna_name'].isin(tk.A2_KNOWN_CDNA)].dropna(subset=['grch38_pos'])
rows = []
for _, v in a2.iterrows():
    pos = int(v['grch38_pos'])
    res = fetch_cadd('7', pos, v['grch38_ref'], v['grch38_alt'])
    rows.append({'cdna_name': v['cdna_name'], 'legacy_name': v['legacy_name'], 'pos': pos,
                 'cadd_phred': res['cadd_phred']})
pd.DataFrame(rows)

,cdna_name,legacy_name,pos,cadd_phred
0,c.3718-2477C>T,3849+10kbC->T,117639961,10.62
1,c.2657+5G>A,2789+5G->A,117602868,23.80
2,c.3140-26A>G,3272-26A->G,117611555,25.00
3,c.2988+1G>A,3120+1G->A,117606754,33.00
4,c.1680-886A>G,1811+1634A->G,117589467,34.00


## Key takeaways

1. **CADD is REAL/live**: `fetch_cadd()` returns a PHRED-scaled score — **≥ 15 ~ top
   3%**, **≥ 20 ~ top 1%** most deleterious.
2. **Training paradigm matters for circularity.** CADD is proxy-trained
   (observed-vs-simulated), *not* on clinical labels — so low **direct** ClinVar
   circularity, but CADD-Splice v1.7 folds in SpliceAI/MMSplice (indirect leakage);
   anchor reproducibility on the **version (v1.7)**, not a training date (tools/10).
3. CFTR is **plus-strand**, so CFTR2's authoritative plus-strand coordinates and alleles
   (merged in benchmark/01) match the coding change directly — no complementing. Pin
   the **genome build** (GRCh38), which is the join key that actually goes wrong.

**Next:** tools/10 — the circularity & temporal-leakage reference.